# 📊 GLOBAL ELECTRONICS SALES DASHBOARD
### Data Cleaning & Preparation Notebook

---

## 1. Project Overview

**Purpose:** This notebook prepares the raw data for the **Global Electronics Sales Dashboard** that will be built in **Power BI**.

**What it does:** The project uses five raw CSV files – **Customers, Exchange Rates, Products, Sales and Stores**. Each file is checked, cleaned and validated one by one using Python (Pandas).

**Why it is needed:** Dashboards are only reliable when the data behind them is correct. Cleaning the data first makes sure that the numbers, dates and categories shown in Power BI are accurate.

**Output/Result:** Five cleaned CSV files (`Customers_Cleaned.csv`, `Exchange_Rates_Cleaned.csv`, `Products_Cleaned.csv`, `Sales_Cleaned.csv`, `Stores_Cleaned.csv`) that are ready to be loaded into Power BI.

**Steps followed for every dataset:**
1. Explore the data (rows, columns, data types)
2. Clean the data (column names, extra spaces, data types)
3. Validate the data (missing values, duplicates, invalid values)
4. Save the cleaned file

## 2. Import Required Libraries

**Purpose:** To load the Python libraries needed for data work.

**What it does:**
- `pandas` – reads CSV files and cleans and analyses tables of data (DataFrames).
- `numpy` – supports number-based operations.
- `matplotlib.pyplot` – used for creating charts.

**Why it is needed:** These libraries must be imported first; every step below depends on them.

**Output/Result:** No visible output. The libraries are now ready to use.

In [62]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


---
## 3. Load the Dataset

**Purpose:** To read all the raw data files into Python.

**What it does:** Five CSV files are loaded into five DataFrames:

| Variable | File | Contains |
|---|---|---|
| `df` | Customers.csv | Customer details (name, gender, location, birthday) |
| `df1` | Exchange_Rates.csv | Daily currency exchange rates |
| `df2` | Products.csv | Product details (brand, category, cost, price) |
| `df3` | Sales.csv | Order and sales transactions |
| `df4` | Stores.csv | Store details (country, state, size, open date) |

The Customers file is read with `encoding="latin-1"` so that special characters in names and places are read correctly.

**Why it is needed:** All five tables together give the full sales picture (who bought, what they bought, where, when and in which currency).

**Output/Result:** No visible output. Data is now available for exploration.

In [63]:
df = pd.read_csv("Customers.csv",encoding="latin-1")
df1 = pd.read_csv("Exchange_Rates.csv")
df2 = pd.read_csv("Products.csv")
df3 = pd.read_csv("Sales.csv")
df4 = pd.read_csv("Stores.csv")

---
## 4. Customers Dataset

### 4.1 Explore the Data

**Purpose:** To understand the Customers table before cleaning it.

**What it does:** Prints the number of rows and columns (`shape`), the first 5 rows (`head`) and a summary of column names, data types and non-null counts (`info`).

**Why it is needed:** This gives a first look at the data and helps us spot problems early (wrong data types, missing values).

**Output/Result:** The table has **15,266 rows and 10 columns**. The `Birthday` column is stored as text, which will need to be fixed later.

In [31]:
print(df.shape)
print(df.head())
print(df.info())

(15266, 10)
   CustomerKey  Gender               Name            City State Code  \
0          301  Female      Lilly Harding  WANDEARAH EAST         SA   
1          325  Female       Madison Hull      MOUNT BUDD         WA   
2          554  Female      Claire Ferres       WINJALLOK        VIC   
3          786    Male  Jai Poltpalingada    MIDDLE RIVER         SA   
4         1042    Male    Aidan Pankhurst   TAWONGA SOUTH        VIC   

               State Zip Code    Country  Continent    Birthday  
0    South Australia     5523  Australia  Australia  07-03-1939  
1  Western Australia     6522  Australia  Australia   9/27/1979  
2           Victoria     3380  Australia  Australia   5/26/1947  
3    South Australia     5223  Australia  Australia   9/17/1957  
4           Victoria     3698  Australia  Australia  11/19/1965  
<class 'pandas.DataFrame'>
RangeIndex: 15266 entries, 0 to 15265
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype
---  ------       ---

### 4.2 Check Duplicates and Missing Values

**Purpose:** To find out whether the data has repeated records or blank values.

**What it does:**
- Counts fully duplicate rows.
- Counts missing values in each column, first as a number and then as a percentage.

**Why it is needed:** Duplicates can inflate customer counts, and missing values can cause incomplete results in the dashboard.

**Output/Result:** There are **no duplicate rows**. Only `State Code` has missing values (**10 rows, about 0.07%**), which is very small.

In [33]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0


In [34]:
print(df.isnull().sum())

CustomerKey     0
Gender          0
Name            0
City            0
State Code     10
State           0
Zip Code        0
Country         0
Continent       0
Birthday        0
dtype: int64


In [36]:
print((df.isnull().sum() / len(df)) * 100)

CustomerKey    0.000000
Gender         0.000000
Name           0.000000
City           0.000000
State Code     0.065505
State          0.000000
Zip Code       0.000000
Country        0.000000
Continent      0.000000
Birthday       0.000000
dtype: float64


### 4.3 Clean Column Names and Text Values

**Purpose:** To remove extra spaces that can cause problems when matching or grouping data.

**What it does:**
- Removes leading and trailing spaces from all column names.
- Removes extra spaces from the text columns (Gender, Name, City, State Code, State, Zip Code, Country, Continent).

**Why it is needed:** Hidden spaces can make the same value look different (for example, `"Male"` and `"Male "`), which breaks filters and slicers in Power BI.

**Output/Result:** Column names and text values are clean and consistent. The list of cleaned text columns is printed.

In [38]:
df.columns = df.columns.str.strip()
print(df.columns)

Index(['CustomerKey', 'Gender', 'Name', 'City', 'State Code', 'State',
       'Zip Code', 'Country', 'Continent', 'Birthday'],
      dtype='str')


In [42]:
text_columns = [
    'Gender', 'Name', 'City', 'State Code',
    'State', 'Zip Code', 'Country', 'Continent'
]

for col in text_columns:
    df[col] = df[col].str.strip()
print(text_columns)

['Gender', 'Name', 'City', 'State Code', 'State', 'Zip Code', 'Country', 'Continent']


### 4.4 Check the Customer ID (CustomerKey)

**Purpose:** To make sure every customer has one unique ID.

**What it does:** Counts duplicate `CustomerKey` values and the number of unique keys.

**Why it is needed:** `CustomerKey` is used to link the Customers table with the Sales table in Power BI. It must be unique.

**Output/Result:** **0 duplicates** and **15,266 unique keys** – equal to the number of rows, so every customer appears only once.

In [43]:
print("Duplicate CustomerKey:", 
      df['CustomerKey'].duplicated().sum())

Duplicate CustomerKey: 0


In [44]:
print("Unique CustomerKey:", df['CustomerKey'].nunique())

Unique CustomerKey: 15266


### 4.5 Fix the Birthday Column

**Purpose:** To convert `Birthday` from text into a proper date format.

**What it does:**
1. Displays the first 20 birthdays and the data type – the dates are text and appear in mixed formats (for example `07-03-1939` and `9/27/1979`).
2. Converts the column to a date type using `pd.to_datetime`. Any value that cannot be converted becomes blank (`errors='coerce'`).
3. Checks how many birthdays are invalid.
4. Displays the birthdays again to confirm the new format.

**Why it is needed:** Correct date types are needed for date-based analysis such as customer age or age groups in Power BI.

**Output/Result:** **0 invalid birthdays**. The column is now a date type and displays in a standard `YYYY-MM-DD` format.

In [64]:
print(df['Birthday'].head(20))
print(df['Birthday'].dtype)

0     07-03-1939
1      9/27/1979
2      5/26/1947
3      9/17/1957
4     11/19/1965
5      1/20/1954
6     11/22/1969
7     03-12-1950
8     12/21/1965
9      7/25/1975
10    08-12-1990
11    10/17/1985
12    03-03-1965
13    11/22/1964
14    07-10-1986
15     9/22/1984
16     9/20/1974
17    10/31/1969
18    12/15/1987
19    04-01-1968
Name: Birthday, dtype: str
str


In [65]:
df['Birthday'] = pd.to_datetime(
    df['Birthday'],
    format='mixed',
    errors='coerce'
)

In [66]:
print("Invalid Birthday:", df['Birthday'].isna().sum())

Invalid Birthday: 0


In [67]:
print(df['Birthday'].head(20))
print(df['Birthday'].dtype)

0    1939-07-03
1    1979-09-27
2    1947-05-26
3    1957-09-17
4    1965-11-19
5    1954-01-20
6    1969-11-22
7    1950-03-12
8    1965-12-21
9    1975-07-25
10   1990-08-12
11   1985-10-17
12   1965-03-03
13   1964-11-22
14   1986-07-10
15   1984-09-22
16   1974-09-20
17   1969-10-31
18   1987-12-15
19   1968-04-01
Name: Birthday, dtype: datetime64[us]
datetime64[us]


### 4.6 Understand Location Data

**Purpose:** To understand where the customers are located.

**What it does:** Counts customers in each country and counts how many unique cities and states exist.

**Why it is needed:** Location is important for geographic charts and maps in the dashboard. This check also shows whether the country values look correct.

**Output/Result:** Customers are from **8 countries**, with the **United States** having the most (6,828). There are **8,258 unique cities** and **512 unique states**.

In [69]:
print(df['Country'].value_counts())

Country
United States     6828
United Kingdom    1944
Canada            1553
Germany           1473
Australia         1420
Netherlands        733
France             670
Italy              645
Name: count, dtype: int64


In [70]:
print("Unique Cities:", df['City'].nunique())
print("Unique States:", df['State'].nunique())

Unique Cities: 8258
Unique States: 512


### 4.7 Final Validation of Customers Data

**Purpose:** To confirm the Customers table is correct after cleaning.

**What it does:** Repeats the duplicate row check and the duplicate `CustomerKey` check, and shows the final shape, first rows and column information.

**Why it is needed:** It confirms that the cleaning steps did not create any new problems.

**Output/Result:** **0 duplicate rows**, **0 duplicate keys**, and the final table still has **15,266 rows and 10 columns** with `Birthday` now as a date.

In [72]:
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate CustomerKey:", 
      df['CustomerKey'].duplicated().sum())

Duplicate rows: 0
Duplicate CustomerKey: 0


In [73]:
print(df.shape)
print(df.head())
print(df.info())

(15266, 10)
   CustomerKey  Gender               Name            City State Code  \
0          301  Female      Lilly Harding  WANDEARAH EAST         SA   
1          325  Female       Madison Hull      MOUNT BUDD         WA   
2          554  Female      Claire Ferres       WINJALLOK        VIC   
3          786    Male  Jai Poltpalingada    MIDDLE RIVER         SA   
4         1042    Male    Aidan Pankhurst   TAWONGA SOUTH        VIC   

               State Zip Code    Country  Continent   Birthday  
0    South Australia     5523  Australia  Australia 1939-07-03  
1  Western Australia     6522  Australia  Australia 1979-09-27  
2           Victoria     3380  Australia  Australia 1947-05-26  
3    South Australia     5223  Australia  Australia 1957-09-17  
4           Victoria     3698  Australia  Australia 1965-11-19  
<class 'pandas.DataFrame'>
RangeIndex: 15266 entries, 0 to 15265
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       

In [74]:
print("Duplicate Rows:", df.duplicated().sum())

print("Duplicate CustomerKey:", 
      df['CustomerKey'].duplicated().sum())

Duplicate Rows: 0
Duplicate CustomerKey: 0


### 4.8 Save the Cleaned Customers File

**Purpose:** To store the cleaned Customers data for use in Power BI.

**What it does:** Saves the table as `Customers_Cleaned.csv` without the index column.

**Why it is needed:** Power BI will load this cleaned file instead of the raw one.

**Output/Result:** `Customers_Cleaned.csv` is created.

In [75]:
df.to_csv('Customers_Cleaned.csv', index=False)

---
## 5. Exchange Rates Dataset

### 5.1 Explore the Data

**Purpose:** To understand the structure of the Exchange Rates table.

**What it does:** Shows the number of rows and columns, the first 5 rows and the data types.

**Why it is needed:** Sales are recorded in different currencies. This table holds the daily rates needed to compare them on a common basis.

**Output/Result:** The table has **11,215 rows and 3 columns** (`Date`, `Currency`, `Exchange`). The `Date` column is stored as text.

In [77]:
print(df1.shape)
print(df1.head())
print(df1.info())

(11215, 3)
       Date Currency  Exchange
0  1/1/2015      USD    1.0000
1  1/1/2015      CAD    1.1583
2  1/1/2015      AUD    1.2214
3  1/1/2015      EUR    0.8237
4  1/1/2015      GBP    0.6415
<class 'pandas.DataFrame'>
RangeIndex: 11215 entries, 0 to 11214
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Date      11215 non-null  str    
 1   Currency  11215 non-null  str    
 2   Exchange  11215 non-null  float64
dtypes: float64(1), str(2)
memory usage: 263.0 KB
None


### 5.2 Clean Column Names, Check Missing Values and Duplicates

**Purpose:** To make sure the table is tidy and complete.

**What it does:**
- Removes extra spaces from column names.
- Checks for missing values.
- Checks for duplicate rows.
- Removes extra spaces from the `Currency` column and counts the records for each currency.

**Why it is needed:** Clean currency codes are needed so they match correctly with the currency codes in the Sales table.

**Output/Result:** **No missing values** and **no duplicate rows**. There are **5 currencies** (USD, CAD, AUD, EUR, GBP), each with 2,243 records.

In [84]:
df1.columns = df1.columns.str.strip()
print(df1.columns)

Index(['Date', 'Currency', 'Exchange'], dtype='str')


In [82]:
print(df1.isnull().sum())

Date        0
Currency    0
Exchange    0
dtype: int64


In [83]:
print("Duplicate Rows:", df1.duplicated().sum())

Duplicate Rows: 0


In [85]:
df1['Currency'] = df1['Currency'].str.strip()

In [87]:
print(df1['Currency'].value_counts())

Currency
USD    2243
CAD    2243
AUD    2243
EUR    2243
GBP    2243
Name: count, dtype: int64


### 5.3 Fix the Date Column

**Purpose:** To convert `Date` from text into a proper date type.

**What it does:** Converts the column using `pd.to_datetime` and then counts invalid dates.

**Why it is needed:** Exchange rates are matched with sales by date, so the column must be a real date.

**Output/Result:** **0 invalid dates**.

In [88]:
df1['Date'] = pd.to_datetime(
    df1['Date'],
    format='mixed',
    errors='coerce'
)

In [90]:
print("Invalid Dates:", df1['Date'].isna().sum())

Invalid Dates: 0


### 5.4 Check Exchange Rate Values and Date Range

**Purpose:** To make sure the exchange rates are realistic.

**What it does:**
- Shows summary statistics of the `Exchange` column (min, max, average, etc.).
- Counts negative and zero exchange rates.
- Finds the earliest and latest dates.

**Why it is needed:** A zero or negative exchange rate would give wrong sales values when converting currencies.

**Output/Result:** Rates range from about **0.63 to 1.73**, with **no negative or zero values**. Dates run from **1 Jan 2015 to 20 Feb 2021**.

In [91]:
print(df1['Exchange'].describe())

count    11215.000000
mean         1.061682
std          0.245519
min          0.628500
25%          0.857800
50%          1.000000
75%          1.311900
max          1.725300
Name: Exchange, dtype: float64


In [92]:
print("Negative Exchange Values:",
      (df1['Exchange'] < 0).sum())

Negative Exchange Values: 0


In [93]:
print("Zero Exchange Values:",
      (df1['Exchange'] == 0).sum())

Zero Exchange Values: 0


In [94]:
print("Minimum Date:", df1['Date'].min())
print("Maximum Date:", df1['Date'].max())

Minimum Date: 2015-01-01 00:00:00
Maximum Date: 2021-02-20 00:00:00


### 5.5 Final Validation of Exchange Rates Data

**Purpose:** To do a last check on the cleaned table.

**What it does:**
- Checks duplicate rows again.
- Checks duplicate **Date + Currency** combinations (each currency should have only one rate per day).
- Shows the final shape, first rows and data types.

**Why it is needed:** If a currency had two rates on the same day, the conversion would be unclear.

**Output/Result:** **0 duplicates**. The table keeps **11,215 rows and 3 columns**, and `Date` is now a date type.

In [95]:
print("Duplicate Rows:",
      df1.duplicated().sum())

Duplicate Rows: 0


In [96]:
print(
    "Duplicate Date-Currency combinations:",
    df1.duplicated(subset=['Date', 'Currency']).sum()
)

Duplicate Date-Currency combinations: 0


In [97]:
print(df1.shape)
print(df1.head())
print(df1.info())

(11215, 3)
        Date Currency  Exchange
0 2015-01-01      USD    1.0000
1 2015-01-01      CAD    1.1583
2 2015-01-01      AUD    1.2214
3 2015-01-01      EUR    0.8237
4 2015-01-01      GBP    0.6415
<class 'pandas.DataFrame'>
RangeIndex: 11215 entries, 0 to 11214
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   Date      11215 non-null  datetime64[us]
 1   Currency  11215 non-null  str           
 2   Exchange  11215 non-null  float64       
dtypes: datetime64[us](1), float64(1), str(1)
memory usage: 263.0 KB
None


In [98]:
print(
    "Duplicate Date-Currency combinations:",
    df1.duplicated(subset=['Date', 'Currency']).sum()
)

Duplicate Date-Currency combinations: 0


### 5.6 Save the Cleaned Exchange Rates File

**Purpose:** To store the cleaned Exchange Rates data for Power BI.

**What it does:** Saves the table as `Exchange_Rates_Cleaned.csv`, prints a success message and shows the final shape and first rows.

**Output/Result:** `Exchange_Rates_Cleaned.csv` is created.

In [100]:
df1.to_csv('Exchange_Rates_Cleaned.csv', index=False)

print("Exchange Rates cleaning completed successfully!")

Exchange Rates cleaning completed successfully!


In [101]:
print(df1.shape)
print(df1.head())

(11215, 3)
        Date Currency  Exchange
0 2015-01-01      USD    1.0000
1 2015-01-01      CAD    1.1583
2 2015-01-01      AUD    1.2214
3 2015-01-01      EUR    0.8237
4 2015-01-01      GBP    0.6415


---
## 6. Products Dataset

### 6.1 Explore the Data

**Purpose:** To understand the Products table.

**What it does:** Shows the number of rows and columns, the first 5 rows and the data types.

**Why it is needed:** This table tells us what is being sold (product, brand, category, cost and price), which is needed for revenue and profit analysis.

**Output/Result:** The table has **2,517 rows and 10 columns**. `Unit Cost USD` and `Unit Price USD` are stored as text (with the `$` sign).

In [102]:
print(df2.shape)
print(df2.head())
print(df2.info())

(2517, 10)
   ProductKey                         Product Name    Brand   Color  \
0           1  Contoso 512MB MP3 Player E51 Silver  Contoso  Silver   
1           2    Contoso 512MB MP3 Player E51 Blue  Contoso    Blue   
2           3     Contoso 1G MP3 Player E100 White  Contoso   White   
3           4    Contoso 2G MP3 Player E200 Silver  Contoso  Silver   
4           5       Contoso 2G MP3 Player E200 Red  Contoso     Red   

  Unit Cost USD Unit Price USD  SubcategoryKey Subcategory  CategoryKey  \
0        $6.62         $12.99              101     MP4&MP3            1   
1        $6.62         $12.99              101     MP4&MP3            1   
2        $7.40         $14.52              101     MP4&MP3            1   
3       $11.00         $21.57              101     MP4&MP3            1   
4       $11.00         $21.57              101     MP4&MP3            1   

  Category  
0    Audio  
1    Audio  
2    Audio  
3    Audio  
4    Audio  
<class 'pandas.DataFrame'>
RangeI

### 6.2 Clean Column Names, Check Missing Values and Duplicates

**Purpose:** To make sure the table is tidy and complete.

**What it does:** Removes extra spaces from column names, checks for missing values and checks for duplicate rows.

**Output/Result:** **No missing values** and **no duplicate rows**.

In [103]:
df2.columns = df2.columns.str.strip()

print(df2.columns)

Index(['ProductKey', 'Product Name', 'Brand', 'Color', 'Unit Cost USD',
       'Unit Price USD', 'SubcategoryKey', 'Subcategory', 'CategoryKey',
       'Category'],
      dtype='str')


In [104]:
print(df2.isnull().sum())

ProductKey        0
Product Name      0
Brand             0
Color             0
Unit Cost USD     0
Unit Price USD    0
SubcategoryKey    0
Subcategory       0
CategoryKey       0
Category          0
dtype: int64


In [106]:
print("Duplicate Rows:", df2.duplicated().sum())

Duplicate Rows: 0


### 6.3 Convert Cost and Price to Numbers

**Purpose:** To change `Unit Cost USD` and `Unit Price USD` from text to numbers.

**What it does:** Removes the `$` sign and commas from both columns and converts them to decimal numbers (`float`). The next cell shows the first rows and the new data types.

**Why it is needed:** Calculations such as revenue, cost and profit are not possible while the values are text.

**Output/Result:** Both columns are now `float64` numbers (for example, 6.62 and 12.99).

In [110]:
df2['Unit Cost USD'] = (
    df2['Unit Cost USD']
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .astype(float)
)

df2['Unit Price USD'] = (
    df2['Unit Price USD']
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .astype(float)
)

In [111]:
print(df2[['Unit Cost USD', 'Unit Price USD']].head())
print(df2[['Unit Cost USD', 'Unit Price USD']].dtypes)

   Unit Cost USD  Unit Price USD
0           6.62           12.99
1           6.62           12.99
2           7.40           14.52
3          11.00           21.57
4          11.00           21.57
Unit Cost USD     float64
Unit Price USD    float64
dtype: object


### 6.4 Check the Product ID (ProductKey)

**Purpose:** To make sure every product has one unique ID.

**What it does:** Counts duplicate `ProductKey` values and the number of unique products.

**Why it is needed:** `ProductKey` links the Products table with the Sales table.

**Output/Result:** **0 duplicates** and **2,517 unique products**.

In [112]:
print("Duplicate ProductKey:",
      df2['ProductKey'].duplicated().sum())

Duplicate ProductKey: 0


In [113]:
print("Unique Products:",
      df2['ProductKey'].nunique())

Unique Products: 2517


### 6.5 Clean Text Columns

**Purpose:** To remove extra spaces from text columns.

**What it does:** Removes extra spaces from Product Name, Brand, Color, Subcategory and Category, then shows the first rows.

**Why it is needed:** Clean text keeps categories and filters consistent in the dashboard.

**Output/Result:** The text columns are clean.

In [115]:
text_columns = [
    'Product Name',
    'Brand',
    'Color',
    'Subcategory',
    'Category'
]

for col in text_columns:
    df2[col] = df2[col].str.strip()


print(df2[text_columns].head())

                          Product Name    Brand   Color Subcategory Category
0  Contoso 512MB MP3 Player E51 Silver  Contoso  Silver     MP4&MP3    Audio
1    Contoso 512MB MP3 Player E51 Blue  Contoso    Blue     MP4&MP3    Audio
2     Contoso 1G MP3 Player E100 White  Contoso   White     MP4&MP3    Audio
3    Contoso 2G MP3 Player E200 Silver  Contoso  Silver     MP4&MP3    Audio
4       Contoso 2G MP3 Player E200 Red  Contoso     Red     MP4&MP3    Audio


### 6.6 Understand Categories and Subcategories

**Purpose:** To understand how products are grouped.

**What it does:**
- Counts products in each category and subcategory.
- Shows the number of unique categories and subcategories.
- Creates a cross-table (`crosstab`) to see which subcategories belong to which category.

**Why it is needed:** Category and subcategory are key fields for product-wise charts and drill-downs in Power BI.

**Output/Result:** There are **8 categories** and **32 subcategories**. *Home Appliances* has the most products (661), followed by *Computers* (606).

In [116]:
print(df2['Category'].value_counts())

Category
Home Appliances                  661
Computers                        606
Cameras and camcorders           372
Cell phones                      285
TV and Video                     222
Games and Toys                   166
Audio                            115
Music, Movies and Audio Books     90
Name: count, dtype: int64


In [117]:
print("Unique Categories:", df2['Category'].nunique())

Unique Categories: 8


In [118]:
print("Unique Subcategories:", df2['Subcategory'].nunique())

Unique Subcategories: 32


In [119]:
print(df2['Subcategory'].value_counts())

Subcategory
Computers Accessories               201
Lamps                               158
Download Games                      120
Projectors & Screens                103
Camcorders                          103
Microwaves                          102
Printers, Scanners & Fax            101
Smart phones & PDAs                 101
Home Theater System                 100
Digital Cameras                     100
Digital SLR Cameras                 100
Home & Office Phones                 92
Movie DVD                            90
Refrigerators                        86
Laptops                              78
Monitors                             78
Fans                                 78
Coffee Machines                      74
Washers & Dryers                     70
Cameras & Camcorders Accessories     69
Touch Screen Phones                  62
Air Conditioners                     62
Bluetooth Headphones                 50
Televisions                          50
Car Video                   

In [120]:
print(pd.crosstab(df2['Category'], df2['Subcategory']))

Subcategory                    Air Conditioners  Bluetooth Headphones  \
Category                                                                
Audio                                         0                    50   
Cameras and camcorders                        0                     0   
Cell phones                                   0                     0   
Computers                                     0                     0   
Games and Toys                                0                     0   
Home Appliances                              62                     0   
Music, Movies and Audio Books                 0                     0   
TV and Video                                  0                     0   

Subcategory                    Boxed Games  Camcorders  \
Category                                                 
Audio                                    0           0   
Cameras and camcorders                   0         103   
Cell phones                          

### 6.7 Check Price and Cost Values

**Purpose:** To make sure the price and cost values make sense.

**What it does:**
- Counts products whose selling price is lower than their cost.
- Shows summary statistics for cost and price.

**Why it is needed:** A price below cost would show wrong profit in the dashboard.

**Output/Result:** **0 products** have a price lower than cost. Cost ranges from **0.48 to 1,060.22** and price from **0.95 to 3,199.99**.

In [121]:
print(
    "Price Lower Than Cost:",
    (df2['Unit Price USD'] < df2['Unit Cost USD']).sum()
)

Price Lower Than Cost: 0


In [122]:
print(df2[['Unit Cost USD', 'Unit Price USD']].describe())

       Unit Cost USD  Unit Price USD
count    2517.000000     2517.000000
mean      147.655562      356.830131
std       184.373683      494.054962
min         0.480000        0.950000
25%        32.250000       69.000000
50%        86.670000      199.990000
75%       183.950000      410.000000
max      1060.220000     3199.990000


### 6.8 Final Validation of Products Data

**Purpose:** To do a last check on the cleaned table.

**What it does:** Shows the data types, checks duplicate rows and duplicate `ProductKey`, and prints the final shape.

**Output/Result:** **0 duplicates**. The final table has **2,517 rows and 10 columns**, and the cost and price columns are numeric.

In [123]:
print(df2.dtypes)

ProductKey          int64
Product Name          str
Brand                 str
Color                 str
Unit Cost USD     float64
Unit Price USD    float64
SubcategoryKey      int64
Subcategory           str
CategoryKey         int64
Category              str
dtype: object


In [124]:
print("Duplicate Rows:", df2.duplicated().sum())

print("Duplicate ProductKey:",
      df2['ProductKey'].duplicated().sum())

Duplicate Rows: 0
Duplicate ProductKey: 0


In [125]:
print("Final Shape:", df2.shape)

Final Shape: (2517, 10)


### 6.9 Save the Cleaned Products File

**Purpose:** To store the cleaned Products data for Power BI.

**What it does:** Saves the table as `Products_Cleaned.csv` and prints a success message.

**Output/Result:** `Products_Cleaned.csv` is created.

In [126]:
df2.to_csv('Products_Cleaned.csv', index=False)

print("Products cleaning completed successfully!")

Products cleaning completed successfully!


---
## 7. Sales Dataset

### 7.1 Explore the Data

**Purpose:** To understand the Sales table, which is the main transaction table of the project.

**What it does:** Shows the number of rows and columns, the first 5 rows and the data types.

**Why it is needed:** This table records every order and links customers, stores and products together.

**Output/Result:** The table has **62,884 rows and 9 columns**. `Order Date` and `Delivery Date` are stored as text.

In [127]:
print(df3.shape)
print(df3.head())
print(df3.info())

(62884, 9)
   Order Number  Line Item Order Date Delivery Date  CustomerKey  StoreKey  \
0        366000          1   1/1/2016           NaN       265598        10   
1        366001          1   1/1/2016     1/13/2016      1269051         0   
2        366001          2   1/1/2016     1/13/2016      1269051         0   
3        366002          1   1/1/2016     1/12/2016       266019         0   
4        366002          2   1/1/2016     1/12/2016       266019         0   

   ProductKey  Quantity Currency Code  
0        1304         1           CAD  
1        1048         2           USD  
2        2007         1           USD  
3        1106         7           CAD  
4         373         1           CAD  
<class 'pandas.DataFrame'>
RangeIndex: 62884 entries, 0 to 62883
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   Order Number   62884 non-null  int64
 1   Line Item      62884 non-null  int64
 2   Order Date

### 7.2 Clean Column Names, Check Missing Values and Duplicates

**Purpose:** To make sure the table is tidy and to find blank values.

**What it does:** Removes extra spaces from column names, counts missing values and checks duplicate rows.

**Output/Result:** **No duplicate rows.** Only `Delivery Date` has missing values (**49,719 rows**). All other columns are complete.

In [128]:
df3.columns = df3.columns.str.strip()

print(df3.columns)

Index(['Order Number', 'Line Item', 'Order Date', 'Delivery Date',
       'CustomerKey', 'StoreKey', 'ProductKey', 'Quantity', 'Currency Code'],
      dtype='str')


In [129]:
print(df3.isnull().sum())

Order Number         0
Line Item            0
Order Date           0
Delivery Date    49719
CustomerKey          0
StoreKey             0
ProductKey           0
Quantity             0
Currency Code        0
dtype: int64


In [130]:
print("Duplicate Rows:", df3.duplicated().sum())

Duplicate Rows: 0


### 7.3 Fix the Date Columns

**Purpose:** To convert `Order Date` and `Delivery Date` from text to date type.

**What it does:** Converts both columns using `pd.to_datetime`, then counts invalid order dates and missing delivery dates.

**Why it is needed:** Correct dates are required for time-based analysis such as monthly and yearly sales trends and delivery time.

**Output/Result:** **0 invalid order dates.** The **49,719 missing delivery dates** are kept as blank (`NaT`) and are not filled in.

In [132]:
df3['Order Date'] = pd.to_datetime(
    df3['Order Date'],
    format='mixed',
    errors='coerce'
)

print("Invalid Order Dates:",
      df3['Order Date'].isna().sum())

Invalid Order Dates: 0


In [134]:
df3['Delivery Date'] = pd.to_datetime(
    df3['Delivery Date'],
    format='mixed',
    errors='coerce'
)

print("Missing Delivery Dates:",
      df3['Delivery Date'].isna().sum())

Missing Delivery Dates: 49719


### 7.4 Check Quantity Values

**Purpose:** To make sure the quantity sold is valid.

**What it does:** Counts rows where `Quantity` is zero or negative.

**Why it is needed:** Zero or negative quantities would give wrong sales totals.

**Output/Result:** **0 zero** and **0 negative** quantities.

In [135]:
print("Zero Quantity:",
      (df3['Quantity'] == 0).sum())

print("Negative Quantity:",
      (df3['Quantity'] < 0).sum())

Zero Quantity: 0
Negative Quantity: 0


### 7.5 Check Currency Codes

**Purpose:** To check the currencies used in the sales.

**What it does:** Counts the orders in each currency, counts unique currencies, removes extra spaces from `Currency Code` and shows the unique values.

**Why it is needed:** `Currency Code` is used to match each sale with the correct exchange rate.

**Output/Result:** There are **5 currencies** – USD (33,767 rows), EUR, GBP, CAD and AUD – the same 5 currencies found in the Exchange Rates table.

In [136]:
print(df3['Currency Code'].value_counts())

Currency Code
USD    33767
EUR    12621
GBP     8140
CAD     5415
AUD     2941
Name: count, dtype: int64


In [137]:
print("Unique Currencies:",
      df3['Currency Code'].nunique())

Unique Currencies: 5


In [143]:
df3['Currency Code'] = df3['Currency Code'].str.strip()

print(df3['Currency Code'].unique())

<StringArray>
['CAD', 'USD', 'GBP', 'EUR', 'AUD']
Length: 5, dtype: str


### 7.6 Check Order and Line Item Uniqueness, and Date Range

**Purpose:** To make sure each order line is recorded only once.

**What it does:**
- Checks duplicate **Order Number + Line Item** combinations.
- Finds the earliest and latest order dates.
- Shows the data types.

**Why it is needed:** One order can have many items, so the pair *Order Number + Line Item* must be unique to avoid double counting sales.

**Output/Result:** **0 duplicates.** Orders run from **1 Jan 2016 to 20 Feb 2021**.

In [139]:
print(
    "Duplicate Order-Line combinations:",
    df3.duplicated(
        subset=['Order Number', 'Line Item']
    ).sum()
)

Duplicate Order-Line combinations: 0


In [140]:
print("Minimum Order Date:",
      df3['Order Date'].min())

print("Maximum Order Date:",
      df3['Order Date'].max())

Minimum Order Date: 2016-01-01 00:00:00
Maximum Order Date: 2021-02-20 00:00:00


In [141]:
print(df3.dtypes)

Order Number              int64
Line Item                 int64
Order Date       datetime64[us]
Delivery Date    datetime64[us]
CustomerKey               int64
StoreKey                  int64
ProductKey                int64
Quantity                  int64
Currency Code               str
dtype: object


### 7.7 Check Keys (Customer, Store, Product) and Quantity

**Purpose:** To check that the ID columns used for linking to other tables are valid.

**What it does:**
- Counts invalid (zero or negative) `CustomerKey`, `StoreKey` and `ProductKey` values.
- Shows summary statistics for `Quantity`.
- Shows the most frequent `StoreKey` values.
- Displays sample rows where `StoreKey` is 0.

**Why it is needed:** Valid keys are needed so that the Sales table connects correctly with the Customers, Stores and Products tables.

**Output/Result:**
- `CustomerKey` and `ProductKey` have **0 invalid values**.
- **13,165 rows have `StoreKey = 0`**. This is not an error – in the Stores table, `StoreKey 0` is the **Online** store, so these are online orders.
- `Quantity` ranges from **1 to 10**.

In [142]:
print(df3.shape)
print(df3.head())
print(df3.info())

(62884, 9)
   Order Number  Line Item Order Date Delivery Date  CustomerKey  StoreKey  \
0        366000          1 2016-01-01           NaT       265598        10   
1        366001          1 2016-01-01    2016-01-13      1269051         0   
2        366001          2 2016-01-01    2016-01-13      1269051         0   
3        366002          1 2016-01-01    2016-01-12       266019         0   
4        366002          2 2016-01-01    2016-01-12       266019         0   

   ProductKey  Quantity Currency Code  
0        1304         1           CAD  
1        1048         2           USD  
2        2007         1           USD  
3        1106         7           CAD  
4         373         1           CAD  
<class 'pandas.DataFrame'>
RangeIndex: 62884 entries, 0 to 62883
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Order Number   62884 non-null  int64         
 1   Line Item      62884 non-

In [144]:
print("Invalid CustomerKey:",
      (df3['CustomerKey'] <= 0).sum())

print("Invalid StoreKey:",
      (df3['StoreKey'] <= 0).sum())

print("Invalid ProductKey:",
      (df3['ProductKey'] <= 0).sum())

Invalid CustomerKey: 0
Invalid StoreKey: 13165
Invalid ProductKey: 0


In [145]:
print(df3['Quantity'].describe())

count    62884.000000
mean         3.144790
std          2.256371
min          1.000000
25%          1.000000
50%          2.000000
75%          4.000000
max         10.000000
Name: Quantity, dtype: float64


In [146]:
print(df3['StoreKey'].value_counts().head(10))

StoreKey
0     13165
9      1577
50     1519
55     1518
54     1498
61     1485
59     1472
45     1471
57     1442
44     1436
Name: count, dtype: int64


In [147]:
print(
    df3[df3['StoreKey'] == 0][
        ['Order Number', 'Order Date', 'Delivery Date',
         'CustomerKey', 'StoreKey', 'ProductKey']
    ].head(10)
)

    Order Number Order Date Delivery Date  CustomerKey  StoreKey  ProductKey
1         366001 2016-01-01    2016-01-13      1269051         0        1048
2         366001 2016-01-01    2016-01-13      1269051         0        2007
3         366002 2016-01-01    2016-01-12       266019         0        1106
4         366002 2016-01-01    2016-01-12       266019         0         373
5         366002 2016-01-01    2016-01-12       266019         0        1080
15        366010 2016-01-01    2016-01-08       370077         0         618
24        366014 2016-01-01    2016-01-06       738549         0         123
25        366014 2016-01-01    2016-01-06       738549         0        1700
31        367005 2016-01-02    2016-01-10       758280         0        1691
32        367005 2016-01-02    2016-01-10       758280         0         319


### 7.8 Check Delivery Date Logic and Other ID Columns

**Purpose:** To confirm the data follows basic business logic.

**What it does:**
- Checks whether any delivery date is earlier than its order date.
- Checks for invalid `Order Number` and `Line Item` values.

**Why it is needed:** A product cannot be delivered before it is ordered. Invalid IDs would point to bad records.

**Output/Result:** **0 rows** have delivery before order, and **0 invalid** order numbers or line items.

In [148]:
print(
    "Delivery Before Order:",
    (
        (df3['Delivery Date'].notna()) &
        (df3['Delivery Date'] < df3['Order Date'])
    ).sum()
)

Delivery Before Order: 0


In [149]:
print(
    "Invalid Order Number:",
    (df3['Order Number'] <= 0).sum()
)

Invalid Order Number: 0


In [150]:
print(
    "Invalid Line Item:",
    (df3['Line Item'] <= 0).sum()
)

Invalid Line Item: 0


### 7.9 Final Validation of Sales Data

**Purpose:** To do a last check on the cleaned Sales table.

**What it does:** Shows the final missing values, data types and shape.

**Output/Result:** The final table has **62,884 rows and 9 columns**. Both date columns are date types. Only `Delivery Date` still has blanks (kept as is).

In [151]:
print(df3.isnull().sum())

Order Number         0
Line Item            0
Order Date           0
Delivery Date    49719
CustomerKey          0
StoreKey             0
ProductKey           0
Quantity             0
Currency Code        0
dtype: int64


In [152]:
print(df3.dtypes)

Order Number              int64
Line Item                 int64
Order Date       datetime64[us]
Delivery Date    datetime64[us]
CustomerKey               int64
StoreKey                  int64
ProductKey                int64
Quantity                  int64
Currency Code               str
dtype: object


In [153]:
print("Final Shape:", df3.shape)

Final Shape: (62884, 9)


### 7.10 Save the Cleaned Sales File

**Purpose:** To store the cleaned Sales data for Power BI.

**What it does:** Saves the table as `Sales_Cleaned.csv`, prints a success message and shows the first rows.

**Output/Result:** `Sales_Cleaned.csv` is created.

In [156]:
df3.to_csv('Sales_Cleaned.csv', index=False)

print("Orders cleaning completed successfully!")

Orders cleaning completed successfully!


In [154]:
print(df3.head())

   Order Number  Line Item Order Date Delivery Date  CustomerKey  StoreKey  \
0        366000          1 2016-01-01           NaT       265598        10   
1        366001          1 2016-01-01    2016-01-13      1269051         0   
2        366001          2 2016-01-01    2016-01-13      1269051         0   
3        366002          1 2016-01-01    2016-01-12       266019         0   
4        366002          2 2016-01-01    2016-01-12       266019         0   

   ProductKey  Quantity Currency Code  
0        1304         1           CAD  
1        1048         2           USD  
2        2007         1           USD  
3        1106         7           CAD  
4         373         1           CAD  


---
## 8. Stores Dataset

### 8.1 Explore the Data

**Purpose:** To understand the Stores table.

**What it does:** Shows the number of rows and columns, the first 5 rows and the data types.

**Why it is needed:** This table tells us where each store is located and how big it is, which is useful for store-wise and country-wise analysis.

**Output/Result:** The table has **67 rows and 5 columns**.

In [157]:
print(df4.shape)
print(df4.head())
print(df4.info())

(67, 5)
   StoreKey    Country                         State  Square Meters  Open Date
0         1  Australia  Australian Capital Territory          595.0   1/1/2008
1         2  Australia            Northern Territory          665.0  1/12/2008
2         3  Australia               South Australia         2000.0   1/7/2012
3         4  Australia                      Tasmania         2000.0   1/1/2010
4         5  Australia                      Victoria         2000.0  12/9/2015
<class 'pandas.DataFrame'>
RangeIndex: 67 entries, 0 to 66
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   StoreKey       67 non-null     int64  
 1   Country        67 non-null     str    
 2   State          67 non-null     str    
 3   Square Meters  66 non-null     float64
 4   Open Date      67 non-null     str    
dtypes: float64(1), int64(1), str(3)
memory usage: 2.7 KB
None


### 8.2 Clean Column Names, Check Missing Values and Duplicates

**Purpose:** To make sure the table is tidy and to find blank or repeated values.

**What it does:** Removes extra spaces from column names, counts missing values, checks duplicate rows and checks duplicate `StoreKey` values.

**Output/Result:** **No duplicate rows or duplicate store keys.** Only `Square Meters` has **1 missing value**.

In [158]:
df4.columns = df4.columns.str.strip()

print(df4.columns)

Index(['StoreKey', 'Country', 'State', 'Square Meters', 'Open Date'], dtype='str')


In [159]:
print(df4.isnull().sum())

StoreKey         0
Country          0
State            0
Square Meters    1
Open Date        0
dtype: int64


In [160]:
print("Duplicate Rows:", df4.duplicated().sum())

Duplicate Rows: 0


In [161]:
print("Duplicate StoreKey:",
      df4['StoreKey'].duplicated().sum())

Duplicate StoreKey: 0


### 8.3 Clean Text Columns and Store Size

**Purpose:** To clean the text columns and make sure store size is a number.

**What it does:**
- Removes extra spaces from `Country` and `State`.
- Converts `Square Meters` to a numeric type and shows summary statistics.

**Why it is needed:** Store size must be numeric to be used in calculations such as sales per square meter.

**Output/Result:** Store sizes range from **245 to 2,105 square meters** (average about 1,402).

In [162]:
text_columns = ['Country', 'State']

for col in text_columns:
    df4[col] = df4[col].str.strip()

print(df4[text_columns].head())

     Country                         State
0  Australia  Australian Capital Territory
1  Australia            Northern Territory
2  Australia               South Australia
3  Australia                      Tasmania
4  Australia                      Victoria


In [163]:
df4['Square Meters'] = pd.to_numeric(
    df4['Square Meters'],
    errors='coerce'
)

print(df4['Square Meters'].describe())

count      66.000000
mean     1402.196970
std       576.404058
min       245.000000
25%      1108.750000
50%      1347.500000
75%      2000.000000
max      2105.000000
Name: Square Meters, dtype: float64


### 8.4 Check Missing and Invalid Store Sizes

**Purpose:** To understand the missing `Square Meters` value and check for invalid sizes.

**What it does:**
- Counts the missing `Square Meters` values and displays that row.
- Counts zero and negative store sizes.

**Why it is needed:** We need to know whether a missing value is a real data problem or expected.

**Output/Result:** The only missing value belongs to **StoreKey 0 (Online)**. An online store has no physical size, so the blank is expected and is kept. There are **no zero or negative sizes**.

In [164]:
print("Missing Square Meters:",
      df4['Square Meters'].isnull().sum())

Missing Square Meters: 1


In [172]:
print(df4[df4['Square Meters'].isna()])

    StoreKey Country   State  Square Meters  Open Date
66         0  Online  Online            NaN 2010-01-01


In [165]:
print("Zero Square Meters:",
      (df4['Square Meters'] == 0).sum())

print("Negative Square Meters:",
      (df4['Square Meters'] < 0).sum())

Zero Square Meters: 0
Negative Square Meters: 0


### 8.5 Fix the Open Date Column

**Purpose:** To convert `Open Date` from text to date type.

**What it does:** Converts the column using `pd.to_datetime`, counts invalid dates and then shows the data types.

**Output/Result:** **0 invalid open dates.** `Open Date` is now a date type.

In [166]:
df4['Open Date'] = pd.to_datetime(
    df4['Open Date'],
    format='mixed',
    errors='coerce'
)

print("Invalid Open Dates:",
      df4['Open Date'].isnull().sum())

Invalid Open Dates: 0


In [167]:
print(df4.dtypes)

StoreKey                  int64
Country                     str
State                       str
Square Meters           float64
Open Date        datetime64[us]
dtype: object


### 8.6 Final Validation of Stores Data

**Purpose:** To do a last check on the cleaned table.

**What it does:** Checks duplicate rows and duplicate `StoreKey` values, and shows the final shape and first rows.

**Output/Result:** **0 duplicates.** The final table has **67 rows and 5 columns**.

In [168]:
print("Duplicate Rows:", df4.duplicated().sum())
print("Duplicate StoreKey:",
      df4['StoreKey'].duplicated().sum())

Duplicate Rows: 0
Duplicate StoreKey: 0


In [169]:
print("Final Shape:", df4.shape)

Final Shape: (67, 5)


In [170]:
print(df4.head())

   StoreKey    Country                         State  Square Meters  Open Date
0         1  Australia  Australian Capital Territory          595.0 2008-01-01
1         2  Australia            Northern Territory          665.0 2008-01-12
2         3  Australia               South Australia         2000.0 2012-01-07
3         4  Australia                      Tasmania         2000.0 2010-01-01
4         5  Australia                      Victoria         2000.0 2015-12-09


### 8.7 Save the Cleaned Stores File

**Purpose:** To store the cleaned Stores data for Power BI.

**What it does:** Saves the table as `Stores_Cleaned.csv` and prints a success message.

**Output/Result:** `Stores_Cleaned.csv` is created.

In [171]:
df4.to_csv('Stores_Cleaned.csv', index=False)

print("Stores cleaning completed successfully!")

Stores cleaning completed successfully!


---
## 9. Final Dataset Summary

All five datasets were explored, cleaned and validated, and saved as separate cleaned CSV files.

| Dataset | Cleaned File | Rows | Columns | Key Cleaning Done |
|---|---|---|---|---|
| Customers | `Customers_Cleaned.csv` | 15,266 | 10 | Removed extra spaces, converted `Birthday` to date |
| Exchange Rates | `Exchange_Rates_Cleaned.csv` | 11,215 | 3 | Removed extra spaces, converted `Date` to date |
| Products | `Products_Cleaned.csv` | 2,517 | 10 | Converted cost and price from text to numbers, removed extra spaces |
| Sales | `Sales_Cleaned.csv` | 62,884 | 9 | Converted `Order Date` and `Delivery Date` to dates |
| Stores | `Stores_Cleaned.csv` | 67 | 5 | Removed extra spaces, converted `Square Meters` and `Open Date` |

**Important checks completed:**
- No duplicate rows in any table.
- Key columns (`CustomerKey`, `ProductKey`, `StoreKey`) are unique in their own tables.
- No invalid dates, no zero or negative quantities and no zero or negative exchange rates.
- No product has a selling price lower than its cost.
- No delivery date is earlier than its order date.

**Things to keep in mind (kept as they are in the data):**
- `State Code` in Customers has 10 blank values (about 0.07%).
- `Delivery Date` in Sales has 49,719 blank values.
- `Square Meters` is blank for the Online store (`StoreKey = 0`), and 13,165 sales rows use this Online store key.

**Readiness for Power BI:** The data types are correct and the tables can be linked using their key columns – `CustomerKey`, `ProductKey` and `StoreKey` link Sales with Customers, Products and Stores, and `Currency Code` with `Date` link Sales with Exchange Rates. The cleaned files are ready to be loaded into Power BI to build the dashboard.